## MDCKII label representations

In [ ]:
1+1

In [ ]:
import tifffile as tiff
import numpy as np
import pandas as pd
import napari
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap

In [ ]:
# Define the root file name (without extension or suffixes)
date= "26.02.10"
file_name_root = "mask_intersect_3"

In [ ]:
# Labels for representation of results
file_name = f"{file_name_root}_noborder.tif"

labels_noborder = tiff.imread(f"./../{date}_MDCKII/masks_intersect/{file_name}")  # shape: (Z, Y, X)

print(f"Shape:     {labels_noborder.shape}")
print(f"Nº cells: {len(np.unique(labels_noborder)) - 1}")
print(f"Dtype:     {labels_noborder.dtype}")
print(f"Memory:   {labels_noborder.nbytes / 1e6:.1f} MB")

In [ ]:
# Read the CSV file with the cell features
data_file = f"{date}_{file_name_root}_properties_quantitative.csv"
cell_features = pd.read_csv(data_file)
df_props = pd.DataFrame(cell_features)

In [ ]:
# Read the CSV file with the cellular clusters obtained by UMAP + HDBSCAN
cluster_file = f"{date}_all_clusters_umap.csv"
clusters = pd.read_csv(cluster_file)
df_clusters = pd.DataFrame(clusters)
df_clusters = df_clusters[df_clusters['source'] == file_name_root] # conserve only the rows of the clusters DataFrame that correspond to the current image
print(len(df_clusters))

# Add the cluster column to the DataFrame of cell properties
df_props_nonan    = df_props.dropna()
print((len(df_props_nonan)))
df_props_clusters = pd.merge(df_props_nonan,
    df_clusters[['label','hdbscan_cluster']], on='label')

print(f"Cells with assigned cluster: {len(df_props_clusters)}")
print(df_props_clusters[['label', 'hdbscan_cluster']].head(10))

## Feature representation

In [ ]:
print(df_props.drop(['label'], axis=1).describe().round(3))

### Histograms

In [ ]:
# Feature distribution, outlier identification, etc.
fig, axes = plt.subplots(6, 4, figsize=(12, 8))
axes = axes.ravel()

df_props['centroid_cell_z_norm_um'].hist(bins=100, ax=axes[0])
axes[0].set_title('Z centroid relative to tissue height (µm)')

df_props['height_um'].hist(bins=100, ax=axes[1])
axes[1].set_title('Cell height (µm)')

df_props['area_apical_um2'].hist(bins=100, ax=axes[2])
axes[2].set_title('Apical area (µm²)')

df_props['area_basal_um2'].hist(bins=100, ax=axes[3])
axes[3].set_title('Basal area (µm²)')

df_props['logratio_ap_bas'].hist(bins=100, ax=axes[4])
axes[4].set_title('LogRatio(Apical/Basal)')

df_props['shape_index_apical'].hist(bins=100, ax=axes[5])
axes[5].set_title('Shape index apical')

df_props['shape_index_basal'].hist(bins=100, ax=axes[6])
axes[6].set_title('Shape index basal')

df_props['volume_um3'].hist(bins=100, ax=axes[7])
axes[7].set_title('Cell volume (µm³)')

df_props['surface_um2'].hist(bins=100, ax=axes[8])
axes[8].set_title('Cell surface area (µm²)')

df_props['sphericity'].hist(bins=100, ax=axes[9])
axes[9].set_title('Sphericity')

df_props['local_density_apical'].hist(bins=100, ax=axes[10])
axes[10].set_title('Local density apical (neighbors within a radius of 75 px)')

df_props['local_density_basal'].hist(bins=100, ax=axes[11])
axes[11].set_title('Local density basal (neighbors within a radius of 75 px)')

df_props['elongation'].hist(bins=100, ax=axes[12])
axes[12].set_title('Elongation')

df_props['angle_z'].hist(bins=100, ax=axes[13])
axes[13].set_title('Vertical orientation (angle with respect to Z axis)')

df_props['curvature_apical_tisular_mean'].hist(bins=100, ax=axes[14])
axes[14].set_title('Mean apical tissue curvature')

df_props['curvature_apical_celular_mean'].hist(bins=100, ax=axes[15])
axes[15].set_title('Mean apical cellular curvature')

df_props['curvature_basal_tisular_inv_mean'].hist(bins=100, ax=axes[16])
axes[16].set_title('Mean basal tissue curvature')

df_props['curvature_basal_celular_inv_mean'].hist(bins=100, ax=axes[17])
axes[17].set_title('Mean basal cellular curvature')

df_props['apical_neighbors_n'].hist(bins=100, ax=axes[18])
axes[18].set_title('Number of apical neighbors')

df_props['basal_neighbors_n'].hist(bins=100, ax=axes[19])
axes[19].set_title('Number of basal neighbors')

df_props['dif_neighbors'].hist(bins=100, ax=axes[20])
axes[20].set_title('Difference in neighbors between apical and basal')

df_props['3D_neighbors_n'].hist(bins=100, ax=axes[21])
axes[21].set_title('Number of 3D neighbors')

df_props_clusters['hdbscan_cluster'].hist(bins=100, ax=axes[22])
axes[22].set_title('Cellular clusters (HDBSCAN)')

plt.tight_layout()
plt.show()

print(df_props.drop(columns=['label']).describe())

### Final representations

In [ ]:
# Proyecciones apical y basal a partir de labels_noborder

tissue_mask_nb = labels_noborder > 0
no_tissue_nb   = ~tissue_mask_nb.any(axis=0)

Y, X   = labels_noborder.shape[1], labels_noborder.shape[2]
ys, xs = np.meshgrid(np.arange(Y), np.arange(X), indexing='ij')

# Apical projection
apical_z_nb    = np.argmax(tissue_mask_nb, axis=0)
apical_labels  = labels_noborder[apical_z_nb, ys, xs].astype(float)
apical_labels[no_tissue_nb] = 0

# Basal projection
flipped_nb     = np.flip(tissue_mask_nb, axis=0)
basal_z_nb     = labels_noborder.shape[0] - 1 - np.argmax(flipped_nb, axis=0)
basal_labels   = labels_noborder[basal_z_nb, ys, xs].astype(float)
basal_labels[no_tissue_nb] = 0

# Random colormap with black background
rng    = np.random.default_rng(seed=67)
n_max  = int(max(apical_labels.max(), basal_labels.max())) + 1
colors = np.vstack([[0, 0, 0], rng.random((n_max, 3))])
cmap   = mcolors.ListedColormap(colors)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(apical_labels.astype(int), cmap=cmap, interpolation='nearest')
axes[0].set_title('Apical projection without borders')
axes[0].axis('off')
axes[1].imshow(basal_labels.astype(int), cmap=cmap, interpolation='nearest')
axes[1].set_title('Basal projection without borders')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
df_props.columns

In [ ]:
# Projections representation, identifying each label with the value of a cellular feature

columna = 'logratio_ap_bas' # test feature

from skimage.segmentation import find_boundaries

def plot_projection(projection, df_props, columna, titulo, cmap_base=plt.cm.RdBu_r, ax=None, mode='percent', boundary_color='yellow', boundary_thickness=1):
    valor_map = df_props.set_index('label')[columna].to_dict()
    n_max     = int(projection.max()) + 1
    valores   = np.array([valor_map.get(i, np.nan) for i in range(n_max)])

    if mode == 'percent':
        vmin, vmax = np.nanpercentile(valores, [1, 99]) # limit the color range to the 1st and 99th percentiles (useful for outlier detection)
    elif mode == 'minmax':
        vmin, vmax = np.nanmin(valores), np.nanmax(valores) # Use the full range of values (useful for neighbors)
    elif mode == 'fixed':
        vmax = np.nanpercentile(valores, 99) 
        vmin = 0 # Helps to highlight positive values in very gradual scales, like tissue curvature
    else:
        raise ValueError("Unknown mode. Use 'percent', 'minmax', or 'fixed'")

    norm   = mcolors.Normalize(vmin=vmin, vmax=vmax)
    colors = cmap_base(norm(valores))
    colors[0]                 = [0, 0, 0, 1]        # fondo negro
    colors[np.isnan(valores)] = [0.3, 0.3, 0.3, 1]  # sin valor → gris

    cmap = mcolors.ListedColormap(colors)

    # Base colored image
    img_colored = colors[projection.astype(int)]  # (H, W, 4)

    # Borders between labels
    boundaries = find_boundaries(projection, mode='inner')  # 1 píxel de grosor
    # Adjust border thickness
    if boundary_thickness > 1:
        from scipy.ndimage import binary_dilation
        boundaries = binary_dilation(boundaries, iterations=boundary_thickness - 1)
    # Border color
    boundary_colors = {'yellow': [1.0, 1.0, 0.0, 1.0],'black' : [0.0, 0.0, 0.0, 1.0],'white' : [1.0, 1.0, 1.0, 1.0]}
    bc = boundary_colors.get(boundary_color, [1.0, 1.0, 0.0, 1.0]) #yellow by default
    img_colored[boundaries] = bc

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    else:
        fig = ax.figure

    ax.imshow(img_colored, interpolation='nearest')
    ax.axis('off')
    ax.set_title(titulo)

    sm = plt.cm.ScalarMappable(cmap=cmap_base, norm=norm)
    sm = plt.cm.ScalarMappable(cmap=cmap_base, norm=norm)
    cb = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02, label=columna)
    cb.ax.tick_params(labelsize=40)      # colorbar numbers size
    cb.set_label(columna, fontsize=14) 

    return ax

# Show the projections of apical and basal labels with the values of the same cellular feature
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_projection(apical_labels, df_props, columna, f'Apical — {columna}', ax=axes[0])
plot_projection(basal_labels,  df_props, columna, f'Basal — {columna}',  ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
df_props.columns

In [ ]:
# Guardar la imagen resultante como jpeg 1024x1024

columnas_fixed = ['curvature_apical_tisular_mean', 'curvature_basal_tisular_inv_mean']
columnas_minmax = ['3D_neighbors_n', 'apical_neighbors_n', 'basal_neighbors_n', 'dif_neighbors']
columnas_percent = df_props.columns.drop(['label'] + columnas_fixed + columnas_minmax)

def save_projection_dual(projection_a, projection_b, df_props, columna, titulo_a, titulo_b, filepath, cmap_base=plt.cm.RdBu_r,mode='percent'):
    fig, axes = plt.subplots(1, 2, figsize=(4096/100, 2048/100), dpi=100)
    plot_projection(projection_a, df_props, columna, titulo_a, cmap_base=cmap_base, ax=axes[0], mode=mode)
    plot_projection(projection_b, df_props, columna, titulo_b, cmap_base=cmap_base, ax=axes[1], mode=mode)
    plt.tight_layout()
    fig.savefig(filepath, dpi=100, format='jpeg',
                bbox_inches='tight', pil_kwargs={'quality': 95})
    plt.close(fig)
    print(f"Guardado: {filepath}")


# Save images
for columna in columnas_percent:
    print(f"Guardando proyecciones para la columna: {columna}")
    save_projection_dual(projection_a= apical_labels, projection_b= basal_labels, df_props= df_props, columna= columna,
                        titulo_a= f'Apical — {columna}', titulo_b= f'Basal — {columna}',
                        filepath= f'./images/{date}_{file_name_root}_{columna}.jpg',
                        mode='percent')
    
for columna in columnas_fixed:
    print(f"Guardando proyecciones para la columna: {columna}")
    save_projection_dual(projection_a= apical_labels, projection_b= basal_labels, df_props= df_props, columna= columna,
                        titulo_a= f'Apical — {columna}', titulo_b= f'Basal — {columna}',
                        filepath= f'./images/{date}_{file_name_root}_{columna}.jpg',
                        mode='fixed')
    
for columna in columnas_minmax:
    print(f"Guardando proyecciones para la columna: {columna}")
    save_projection_dual(projection_a= apical_labels, projection_b= basal_labels, df_props= df_props, columna= columna,
                        titulo_a= f'Apical — {columna}', titulo_b= f'Basal — {columna}',
                        filepath= f'./images/{date}_{file_name_root}_{columna}.jpg',
                        mode='minmax')

cmap = ListedColormap(["#9e9e9e", "#FE8E2F","#2257ab"])
save_projection_dual(projection_a= apical_labels, projection_b= basal_labels, df_props= df_props_clusters, columna= 'hdbscan_cluster',
                        titulo_a= 'Apical clusters', titulo_b= 'Basal clusters',
                        cmap_base=cmap,
                        filepath= f'./images/{date}_{file_name_root}_hdbscan_cluster.jpg',
                        mode='minmax')

In [ ]:
# Some variables need a different plotting system
# Show the projections of apical and basal labels with the values of different cellular features (the apical and basal variant of the same feature)

def plot_projection2(projection, df_props, columna, titulo,
                    cmap_base=plt.cm.RdBu_r,
                    ax=None,
                    norm=None,
                    boundary_color='yellow',
                    boundary_thickness=1):

    valor_map = df_props.set_index('label')[columna].to_dict()

    n_max   = int(projection.max()) + 1
    valores = np.array([valor_map.get(i, np.nan) for i in range(n_max)])

    if norm is None:
        norm = mcolors.Normalize(vmin=np.nanpercentile(valores, 1),
            vmax=np.nanpercentile(valores, 99))

    colors = cmap_base(norm(valores))

    colors[0]                 = [0, 0, 0, 1]
    colors[np.isnan(valores)] = [0.3, 0.3, 0.3, 1]

    img_colored = colors[projection.astype(int)]

    boundaries = find_boundaries(projection, mode='inner')

    if boundary_thickness > 1:
        from scipy.ndimage import binary_dilation
        boundaries = binary_dilation(boundaries,iterations=boundary_thickness - 1)

    boundary_colors = {'yellow': [1, 1, 0, 1],'black' : [0, 0, 0, 1],'white' : [1, 1, 1, 1]}

    img_colored[boundaries] = boundary_colors.get(boundary_color,[1, 1, 0, 1])

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    ax.imshow(img_colored, interpolation='nearest')
    ax.axis('off')
    ax.set_title(titulo)

    sm = plt.cm.ScalarMappable(cmap=cmap_base, norm=norm)
    cb = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02,label=columna)
    cb.ax.tick_params(labelsize=40)      # colorbar numbers size

    return ax


## Common range for both projections

# columna1= 'curvature_apical_celular_mean'
# columna2= 'curvature_basal_celular_inv_mean'
columna1= 'curvature_apical_tisular_mean'
columna2= 'curvature_basal_tisular_inv_mean'
# columna1= 'shape_index_apical'
# columna2= 'shape_index_basal'
# columna1= 'local_density_apical'
# columna2= 'local_density_basal'
# columna1= 'local_density_apical'
# columna2= 'local_density_basal'
# columna1= 'apical_neighbors_n'
# columna2= 'basal_neighbors_n'
# columna1= 'area_apical_um2'
# columna2= 'area_basal_um2'

# Normalization shared: joint percentiles
vals_ap = df_props[f'{columna1}'].values
vals_ba = df_props[f'{columna2}'].values
vals_all = np.concatenate([vals_ap, vals_ba])

vmin = 0 #for curvatures # for curvatures
# vmin = np.min(vals_all) # for neighbors
# vmax = np.max(vals_all) # for neighbors
# vmin = np.percentile(vals_all, 1)
vmax = np.percentile(vals_all, 99)

norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

print(f'Rango común: {vmin:.3f} – {vmax:.3f}')

# Figure representation
fig, axes = plt.subplots(1, 2, figsize=(4096/100, 2048/100), dpi=100)
plot_projection2(apical_labels,df_props,columna1,f'Apical — {columna1}',norm=norm,ax=axes[0])
plot_projection2(basal_labels,df_props,columna2,f'Basal — {columna2}',norm=norm,ax=axes[1])

plt.tight_layout()
fig.savefig(fname= f'./images/{date}_{file_name_root}_{columna1}_{columna2}.jpg',dpi=100,format='jpeg',bbox_inches='tight',pil_kwargs={'quality': 95})
plt.show()